# Module 08: Campaign Optimization, Recommendation & Evaluation
### Member 4: Optimization, Recommendation & Web Dashboard

This notebook implements the optimization and recommendation engine for marketing campaigns.
It integrates the ML models trained by the team:
- **ROI Model**: Tuned Random Forest Regressor (Member 1)
- **Revenue Model**: Tuned Gradient Boosting Regressor (Member 2)
- **Conversion Model**: Gradient Boosting Regressor with log1p (Member 2)
- **Success Model**: Gradient Boosting Classifier (Member 3)
- **Explainability Insights**: SHAP Tree Explainer (Member 3)

The optimization engine evaluates candidate campaign variations and ranks them using a documented multi-objective composite score.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Ensure project root is in path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.predictor_engine import predict_campaign, get_all_models_status
from src.campaign_optimizer import optimize_campaign
from src.dataset_analytics import get_sample_campaign, get_categorical_options

print("Optimization modules and models imported successfully!")

In [2]:
# Verify all 4 team models status
status = get_all_models_status()
print("Loaded Models Status:")
for model_name, info in status.items():
    if model_name != "status":
        print(f" - {model_name}: {info}")

In [3]:
# Define a test campaign configuration
test_campaign = get_sample_campaign("standard")
print("Initial Baseline Campaign Configuration:")
for k, v in list(test_campaign.items())[:12]:
    print(f"  {k}: {v}")

In [4]:
# Run baseline prediction across all 4 models
baseline_pred = predict_campaign(test_campaign)
print("Baseline Model Predictions:")
print(f"  Predicted ROI: {baseline_pred['predicted_roi_formatted']}")
print(f"  Predicted Revenue: {baseline_pred['predicted_revenue_formatted']}")
print(f"  Predicted Conversions: {baseline_pred['predicted_conversions']}")
print(f"  Conversion Rate: {baseline_pred['predicted_conversion_rate_formatted']}")
print(f"  Success Probability: {baseline_pred['success_probability_formatted']}")

In [5]:
# Execute AI Campaign Optimization across variations
opt_results = optimize_campaign(test_campaign, priority="balanced")

rec = opt_results["recommended"]
deltas = opt_results["deltas"]

print("=" * 60)
print("AI RECOMMENDED CAMPAIGN:")
print("=" * 60)
print(f"Strategy: {rec['campaign_name']}")
print(f"Composite Optimization Score: {rec['composite_score']}/100")
print(f"Recommended ROI: {rec['predicted_roi_formatted']} (Δ {deltas['roi_diff_formatted']})")
print(f"Recommended Revenue: {rec['predicted_revenue_formatted']} (Δ {deltas['revenue_diff_formatted']})")
print(f"Conversion Rate: {rec['predicted_conversion_rate_formatted']} (Δ {deltas['conversion_rate_diff_formatted']})")
print(f"Success Probability: {rec['success_probability_formatted']}")

print("\nExplainability Insights:")
for idx, exp in enumerate(opt_results["explainability"]):
    print(f" {idx+1}. [{exp['shap_rank']}] {exp['feature']}: {exp['explanation']}")

In [6]:
# Display All Evaluated Strategic Alternatives
ranked_df = pd.DataFrame([
    {
        "Strategy": c["campaign_name"],
        "Platform": c["config"]["platform"],
        "Format": c["config"]["creative_format"],
        "Device": c["config"]["device_type"],
        "ROI (%)": c["predicted_roi"],
        "Revenue (INR)": c["predicted_revenue"],
        "Success Prob (%)": c["success_probability_percent"],
        "Score": c["composite_score"]
    }
    for c in opt_results["all_ranked_candidates"]
])
display(ranked_df)